In [1]:
import os, glob, random, json
from dataclasses import dataclass
from typing import List, Tuple

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# ----------------------------
# Reproducibility
# ----------------------------
def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

SEED = 24
seed_everything(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

# ----------------------------
# Config
# ----------------------------
@dataclass
class CFG:
    DATA_ROOT: str = "/kaggle/input/sen2fire/Sen2Fire/Sen2Fire"
    SCENES: Tuple[str, ...] = ("scene1", "scene2", "scene3", "scene4")
    PATCH_EXT: str = ".npz"

    # Global split (VD default)
    USE_GLOBAL_SPLIT: bool = True
    GLOBAL_TRAIN_RATIO: float = 0.80
    GLOBAL_VAL_RATIO: float = 0.10
    GLOBAL_TEST_RATIO: float = 0.10
    KEEP_VAL_TEST_NATURAL: bool = True

    # Keys inside npz
    X_KEY: str = "image"     # (12,512,512)
    A_KEY: str = "aerosol"   # (512,512) -> becomes 1 channel
    Y_KEY: str = "label"     # (512,512)

    # Fire patch definition
    FIRE_PATCH_MIN_RATIO: float = 0

    # Controlled train pool (VD default)
    USE_CONTROLLED_POOL: bool = True
    POOL_KEEP_FIRE: int = -1
    NONFIRE_PER_FIRE: int = 3

    # Training config (RAM-safe)
    IN_CHANNELS: int = 13
    H: int = 512
    W: int = 512
    BATCH_SIZE: int = 2
    NUM_WORKERS: int = 2
    LR: float = 1e-4
    EPOCHS: int = 20 

    VERBOSE: bool = True

cfg = CFG()

assert abs((cfg.GLOBAL_TRAIN_RATIO + cfg.GLOBAL_VAL_RATIO + cfg.GLOBAL_TEST_RATIO) - 1.0) < 1e-6
assert os.path.exists(cfg.DATA_ROOT), f"DATA_ROOT not found: {cfg.DATA_ROOT}"
print("DATA_ROOT OK:", cfg.DATA_ROOT)


Device: cuda
DATA_ROOT OK: /kaggle/input/sen2fire/Sen2Fire/Sen2Fire


In [2]:
def list_scene_files(data_root: str, scene_name: str, ext: str = ".npz") -> List[str]:
    scene_dir = os.path.join(data_root, scene_name)
    return sorted(glob.glob(os.path.join(scene_dir, f"*{ext}")))

scene_to_files = {}
total = 0
for s in cfg.SCENES:
    files = list_scene_files(cfg.DATA_ROOT, s, cfg.PATCH_EXT)
    scene_to_files[s] = files
    total += len(files)
    print(f"{s}: {len(files)} patches")

print("Total patches:", total)
if total == 0:
    raise RuntimeError("No patch files found. Check DATA_ROOT / PATCH_EXT.")


scene1: 864 patches
scene2: 594 patches
scene3: 504 patches
scene4: 504 patches
Total patches: 2466


In [3]:
def inspect_npz(npz_path: str):
    with np.load(npz_path) as data:
        return {k: (data[k].shape, str(data[k].dtype)) for k in data.keys()}

sample_path = scene_to_files[cfg.SCENES[0]][0]
print("Sample file:", sample_path)
info = inspect_npz(sample_path)
for k, (shape, dtype) in info.items():
    print(f"  - {k:>10s}: shape={shape}, dtype={dtype}")

for rk in [cfg.X_KEY, cfg.A_KEY, cfg.Y_KEY]:
    if rk not in info:
        raise KeyError(f"Missing key '{rk}' in npz. Found keys: {list(info.keys())}")


Sample file: /kaggle/input/sen2fire/Sen2Fire/Sen2Fire/scene1/scene_1_patch_10_1.npz
  -      image: shape=(12, 512, 512), dtype=int16
  -    aerosol: shape=(512, 512), dtype=float32
  -      label: shape=(512, 512), dtype=uint8


In [4]:
rows = []
for s, files in scene_to_files.items():
    for p in files:
        rows.append({"scene": s, "path": p})
manifest = pd.DataFrame(rows)

def fire_ratio_from_path(npz_path: str) -> float:
    with np.load(npz_path) as data:
        y = data[cfg.Y_KEY]
        if y.ndim == 3 and y.shape[-1] == 1:
            y = y[..., 0]
        yb = (y > 0).astype(np.uint8)
        return float(yb.mean())

fire_ratios = []
has_fire = []
for p in manifest["path"].tolist():
    r = fire_ratio_from_path(p)
    fire_ratios.append(r)
    has_fire.append(1 if r > cfg.FIRE_PATCH_MIN_RATIO else 0)

manifest["fire_ratio"] = fire_ratios
manifest["has_fire"] = has_fire

print("\nFULL dataset has_fire distribution:")
print(manifest["has_fire"].value_counts().sort_index())
print("\nFULL dataset has_fire ratio:")
print(manifest["has_fire"].value_counts(normalize=True).sort_index())



FULL dataset has_fire distribution:
has_fire
0    2117
1     349
Name: count, dtype: int64

FULL dataset has_fire ratio:
has_fire
0    0.858475
1    0.141525
Name: proportion, dtype: float64


# DATASET PREPARATION

In [5]:
def stratified_split(df, train_ratio, val_ratio, seed=42):
    df = df.sample(frac=1.0, random_state=seed).reset_index(drop=True)
    parts = []
    for cls in [0, 1]:
        sub = df[df["has_fire"] == cls].copy()
        n = len(sub)
        n_train = int(round(n * train_ratio))
        n_val   = int(round(n * val_ratio))
        sub_train = sub.iloc[:n_train]
        sub_val   = sub.iloc[n_train:n_train+n_val]
        sub_test  = sub.iloc[n_train+n_val:]
        parts.append((sub_train, sub_val, sub_test))

    train_df = pd.concat([parts[0][0], parts[1][0]]).sample(frac=1.0, random_state=seed).reset_index(drop=True)
    val_df   = pd.concat([parts[0][1], parts[1][1]]).sample(frac=1.0, random_state=seed).reset_index(drop=True)
    test_df  = pd.concat([parts[0][2], parts[1][2]]).sample(frac=1.0, random_state=seed).reset_index(drop=True)
    return train_df, val_df, test_df

train_pool_df, val_df, test_df = stratified_split(
    manifest,
    train_ratio=cfg.GLOBAL_TRAIN_RATIO,
    val_ratio=cfg.GLOBAL_VAL_RATIO,
    seed=SEED
)

print("\nGLOBAL split counts:")
print("  Train pool:", len(train_pool_df))
print("  Val      :", len(val_df))
print("  Test     :", len(test_df))

# Controlled pool applies to TRAIN only
if cfg.USE_CONTROLLED_POOL:
    fire_df   = train_pool_df[train_pool_df["has_fire"] == 1].copy()
    nofire_df = train_pool_df[train_pool_df["has_fire"] == 0].copy()

    n_fire_total = len(fire_df)
    n_nofire_total = len(nofire_df)

    n_fire_keep = n_fire_total if cfg.POOL_KEEP_FIRE in (-1, None) else min(cfg.POOL_KEEP_FIRE, n_fire_total)
    fire_keep = fire_df.sample(n=n_fire_keep, random_state=SEED) if n_fire_keep > 0 else fire_df.iloc[:0]

    n_nofire_keep = min(n_nofire_total, n_fire_keep * int(cfg.NONFIRE_PER_FIRE))
    nofire_keep = nofire_df.sample(n=n_nofire_keep, random_state=SEED) if n_nofire_keep > 0 else nofire_df.iloc[:0]

    train_df = pd.concat([fire_keep, nofire_keep]).sample(frac=1.0, random_state=SEED).reset_index(drop=True)

    print("\nTRAIN controlled sampling:")
    print(f"  fire_total_in_pool   : {n_fire_total}")
    print(f"  nofire_total_in_pool : {n_nofire_total}")
    print(f"  fire_kept            : {len(fire_keep)}")
    print(f"  nofire_kept          : {len(nofire_keep)} (NONFIRE_PER_FIRE={cfg.NONFIRE_PER_FIRE})")
    print(f"  train_final          : {len(train_df)}")
else:
    train_df = train_pool_df.copy()

train_paths = train_df["path"].tolist()
val_paths   = val_df["path"].tolist()
test_paths  = test_df["path"].tolist()

# leakage check
assert len(set(train_paths)&set(val_paths))==0
assert len(set(train_paths)&set(test_paths))==0
assert len(set(val_paths)&set(test_paths))==0

print("\nFinal used split sizes:")
print("  train:", len(train_paths))
print("  val  :", len(val_paths))
print("  test :", len(test_paths))



GLOBAL split counts:
  Train pool: 1973
  Val      : 247
  Test     : 246

TRAIN controlled sampling:
  fire_total_in_pool   : 279
  nofire_total_in_pool : 1694
  fire_kept            : 279
  nofire_kept          : 837 (NONFIRE_PER_FIRE=3)
  train_final          : 1116

Final used split sizes:
  train: 1116
  val  : 247
  test : 246


In [6]:
class Sen2FireDataset(Dataset):
    def __init__(self, paths: List[str], with_label: bool = True):
        self.paths = paths
        self.with_label = with_label

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        p = self.paths[idx]
        with np.load(p) as d:
            img12 = d[cfg.X_KEY].astype(np.float32)      # (12,512,512)
            aer   = d[cfg.A_KEY].astype(np.float32)[None, ...]  # (1,512,512)
            x = np.concatenate([img12, aer], axis=0)     # (13,512,512)

            if self.with_label:
                y = d[cfg.Y_KEY]
                if y.ndim == 2:
                    y = y[None, ...]
                y = (y > 0).astype(np.float32)
                return torch.from_numpy(x), torch.from_numpy(y)
            else:
                return torch.from_numpy(x)


In [7]:
train_loader = DataLoader(Sen2FireDataset(train_paths, with_label=True),
                          batch_size=cfg.BATCH_SIZE, shuffle=True,
                          num_workers=cfg.NUM_WORKERS, pin_memory=True)

val_loader = DataLoader(Sen2FireDataset(val_paths, with_label=True),
                        batch_size=cfg.BATCH_SIZE, shuffle=False,
                        num_workers=cfg.NUM_WORKERS, pin_memory=True)

test_loader = DataLoader(Sen2FireDataset(test_paths, with_label=True),
                         batch_size=cfg.BATCH_SIZE, shuffle=False,
                         num_workers=cfg.NUM_WORKERS, pin_memory=True)


In [8]:
@torch.no_grad()
def compute_mean_std(loader, max_batches=50):
    mean = torch.zeros(cfg.IN_CHANNELS, device=DEVICE)
    var  = torch.zeros(cfg.IN_CHANNELS, device=DEVICE)
    n_batches = 0

    for bi, (x, y) in enumerate(loader):
        if bi >= max_batches:
            break
        x = x.to(DEVICE)  # (B,C,H,W)
        x_ = x.view(x.size(0), cfg.IN_CHANNELS, -1)
        mean += x_.mean(dim=(0,2))
        var  += x_.var(dim=(0,2), unbiased=False)
        n_batches += 1

    mean /= max(n_batches, 1)
    var  /= max(n_batches, 1)
    std = torch.sqrt(var + 1e-6)
    return mean.detach().cpu().tolist(), std.detach().cpu().tolist()

MEAN_13, STD_13 = compute_mean_std(train_loader, max_batches=50)
print("MEAN_13 len:", len(MEAN_13), "STD_13 len:", len(STD_13))


MEAN_13 len: 13 STD_13 len: 13


# MODEL PREPARATION


In [9]:
!pip -q install segmentation-models-pytorch


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 5.2 MB/s eta 0:00:00


In [10]:
def normalize_batch(x, mean_13, std_13):
    mean_t = torch.tensor(mean_13, device=x.device).view(1, cfg.IN_CHANNELS, 1, 1)
    std_t  = torch.tensor(std_13,  device=x.device).view(1, cfg.IN_CHANNELS, 1, 1)
    return (x - mean_t) / (std_t + 1e-6)

In [11]:
import segmentation_models_pytorch as smp

criterion = nn.BCEWithLogitsLoss()
def build_unet(in_channels=13):
    model = smp.Unet(
        encoder_name="efficientnet-b4",
        encoder_weights="imagenet",
        in_channels=in_channels,
        classes=1,
        activation=None
    )
    return model
    
model = build_unet(cfg.IN_CHANNELS).to(DEVICE)
print("Model:", type(model).__name__)

/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

config.json:   0%|          | 0.00/106 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/77.9M [00:00<?, ?B/s]

Model: Unet


In [12]:
from torch.amp import autocast, GradScaler
from sklearn.metrics import average_precision_score
from sklearn.metrics import roc_auc_score
import time

optimizer = torch.optim.Adam(model.parameters(), lr=cfg.LR)
scaler = GradScaler(enabled=(DEVICE.type == "cuda"))

def train_one_epoch(model, loader, thr=0.5, eps=1e-6):
    model.train()
    total_loss_sum = 0.0
    total_samples = 0

    total_tp = total_fp = total_fn = total_tn = 0.0
    all_probs, all_targets = [], []

    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        x = normalize_batch(x, MEAN_13, STD_13)

        optimizer.zero_grad(set_to_none=True)

        with autocast("cuda", enabled=(DEVICE.type == "cuda")):
            logits = model(x)
            loss = criterion(logits, y)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        bs = x.size(0)
        total_loss_sum += loss.item() * bs
        total_samples += bs

        probs = torch.sigmoid(logits)
        preds = (probs >= thr).float()

        total_tp += (preds * y).sum().item()
        total_fp += (preds * (1 - y)).sum().item()
        total_fn += ((1 - preds) * y).sum().item()
        total_tn += ((1 - preds) * (1 - y)).sum().item()

        all_probs.append(probs.flatten().detach().cpu())
        all_targets.append(y.flatten().detach().cpu())

    precision = (total_tp + eps) / (total_tp + total_fp + eps)
    recall    = (total_tp + eps) / (total_tp + total_fn + eps)
    f1        = (2 * precision * recall) / (precision + recall + eps)
    acc       = (total_tp + total_tn + eps) / (
        total_tp + total_tn + total_fp + total_fn + eps
    )

    all_probs = torch.cat(all_probs).numpy()
    all_targets = torch.cat(all_targets).numpy()
    auc = average_precision_score(all_targets, all_probs)

    loss_global = total_loss_sum / total_samples

    return loss_global, precision, recall, f1, acc, auc


@torch.no_grad()
def validate_epoch(model, loader, thr=0.5, eps=1e-6):
    model.eval()
    total_loss_sum = 0.0
    total_samples = 0

    total_tp = total_fp = total_fn = total_tn = 0.0
    all_probs, all_targets = [], []

    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        x = normalize_batch(x, MEAN_13, STD_13)

        logits = model(x)
        loss = criterion(logits, y)

        bs = x.size(0)
        total_loss_sum += loss.item() * bs
        total_samples += bs

        probs = torch.sigmoid(logits)
        preds = (probs >= thr).float()

        total_tp += (preds * y).sum().item()
        total_fp += (preds * (1 - y)).sum().item()
        total_fn += ((1 - preds) * y).sum().item()
        total_tn += ((1 - preds) * (1 - y)).sum().item()

        all_probs.append(probs.flatten().cpu())
        all_targets.append(y.flatten().cpu())

    precision = (total_tp + eps) / (total_tp + total_fp + eps)
    recall    = (total_tp + eps) / (total_tp + total_fn + eps)
    f1        = (2 * precision * recall) / (precision + recall + eps)
    acc       = (total_tp + total_tn + eps) / (
        total_tp + total_tn + total_fp + total_fn + eps
    )

    all_probs = torch.cat(all_probs).numpy()
    all_targets = torch.cat(all_targets).numpy()
    auc = average_precision_score(all_targets, all_probs)

    loss_global = total_loss_sum / total_samples

    return loss_global, precision, recall, f1, acc, auc

# TRAINING

In [20]:
BEST_PATH = "/kaggle/working/efficientb4.pth"
best_val_f1 = -1

import pandas as pd
history = []

for epoch in range(1, cfg.EPOCHS + 1):
    t0 = time.time()

    tr_loss, tr_prec, tr_rec, tr_f1, tr_acc, tr_auc = train_one_epoch(model, train_loader)
    va_loss, va_prec, va_rec, va_f1, va_acc, va_auc = validate_epoch(model, val_loader, thr=0.5)
    elapsed = time.time() - t0

    print(
        f"[unet_basic] Epoch {epoch:03d} | "
        f"tr_loss={tr_loss:.4f} tr_f1={tr_f1:.4f} tr_acc={tr_acc:.4f} tr_PRauc={tr_auc:.4f} || "
        f"va_loss={va_loss:.4f} va_f1={va_f1:.4f} va_acc={va_acc:.4f} va_PRauc={va_auc:.4f} | "
        f"{elapsed:.1f}s"
    )

    history.append({
        "epoch": epoch,
    
        "tr_loss": tr_loss,
        "tr_precision": tr_prec,
        "tr_recall": tr_rec,
        "tr_f1": tr_f1,
        "tr_acc": tr_acc,
        "tr_PRauc": tr_auc,
    
        "va_loss": va_loss,
        "va_precision": va_prec,
        "va_recall": va_rec,
        "va_f1": va_f1,
        "va_acc": va_acc,
        "va_PRauc": va_auc,
    
        "seconds": elapsed
    })

    if va_f1 > best_val_f1:
        torch.save({
            "model_state": model.state_dict(),
            "mean_13": MEAN_13,
            "std_13": STD_13,
            "epoch": epoch,
            "best_val_f1": best_val_f1,
            "va_auc": va_auc,
        }, BEST_PATH)


[unet_basic] Epoch 001 | tr_loss=0.1889 tr_f1=0.1420 tr_acc=0.9428 tr_PRauc=0.2755 || va_loss=0.1381 va_f1=0.0651 va_acc=0.9519 va_PRauc=0.3289 | 326.0s
[unet_basic] Epoch 002 | tr_loss=0.1639 tr_f1=0.3168 tr_acc=0.9467 tr_PRauc=0.4100 || va_loss=0.1232 va_f1=0.0018 va_acc=0.9522 va_PRauc=0.4055 | 324.8s
[unet_basic] Epoch 003 | tr_loss=0.1356 tr_f1=0.4874 tr_acc=0.9551 tr_PRauc=0.5765 || va_loss=0.1477 va_f1=0.0000 va_acc=0.9522 va_PRauc=0.3395 | 325.0s
[unet_basic] Epoch 004 | tr_loss=0.1011 tr_f1=0.6632 tr_acc=0.9653 tr_PRauc=0.7327 || va_loss=0.1086 va_f1=0.0798 va_acc=0.9539 va_PRauc=0.6033 | 323.2s
[unet_basic] Epoch 005 | tr_loss=0.0822 tr_f1=0.7421 tr_acc=0.9721 tr_PRauc=0.8072 || va_loss=0.1484 va_f1=0.0000 va_acc=0.9522 va_PRauc=0.5431 | 323.6s
[unet_basic] Epoch 006 | tr_loss=0.0665 tr_f1=0.7983 tr_acc=0.9774 tr_PRauc=0.8634 || va_loss=0.1328 va_f1=0.0000 va_acc=0.9522 va_PRauc=0.6381 | 323.4s
[unet_basic] Epoch 007 | tr_loss=0.0569 tr_f1=0.8356 tr_acc=0.9813 tr_PRauc=0.8944

In [21]:
df_history = pd.DataFrame(history)

df_history = df_history.round(4)
csv_path = "/kaggle/working/training_metrics_eff1.csv"
df_history.to_csv(csv_path, index=False)
print("Saved to:", csv_path)
df_history

Saved to: /kaggle/working/training_metrics_eff1.csv


,epoch,tr_loss,tr_precision,tr_recall,tr_f1,tr_acc,tr_PRauc,va_loss,va_precision,va_recall,va_f1,va_acc,va_PRauc,seconds
0,1,0.1889,0.6055,0.0805,0.1420,0.9428,0.2755,0.1381,0.4598,0.0350,0.0651,0.9519,0.3289,326.0277
1,2,0.1639,0.6474,0.2097,0.3168,0.9467,0.4100,0.1232,0.7221,0.0009,0.0018,0.9522,0.4055,324.7601
2,3,0.1356,0.7453,0.3621,0.4874,0.9551,0.5765,0.1477,1.0000,0.0000,0.0000,0.9522,0.3395,324.9510
3,4,0.1011,0.7751,0.5795,0.6632,0.9653,0.7327,0.1086,0.8705,0.0418,0.0798,0.9539,0.6033,323.1811
4,5,0.0822,0.8130,0.6825,0.7421,0.9721,0.8072,0.1484,1.0000,0.0000,0.0000,0.9522,0.5431,323.6256
5,6,0.0665,0.8429,0.7582,0.7983,0.9774,0.8634,0.1328,0.0000,0.0000,0.0000,0.9522,0.6381,323.4198
6,7,0.0569,0.8666,0.8068,0.8356,0.9813,0.8944,0.1106,0.8246,0.1868,0.3046,0.9592,0.6277,322.1071
7,8,0.0497,0.8725,0.8349,0.8533,0.9831,0.9135,0.1192,0.9612,0.0966,0.1756,0.9566,0.7063,324.6199
8,9,0.0411,0.8916,0.8690,0.8801,0.9861,0.9351,0.0982,0.8344,0.3501,0.4932,0.9656,0.6908,323.9097
9,10,0.0340,0.9098,0.8882,0.8989,0.9882,0.9565,0.1098,0.8758,0.2830,0.4278,0.9638,0.7042,320.1268


In [22]:
@torch.no_grad()
def find_best_thr_global_f1(model, loader, grid=None, eps=1e-6):
    model.eval()
    if grid is None:
        grid = np.linspace(0.05, 0.95, 19)

    best_thr = None
    best_f1 = -1
    log = []

    for thr in grid:
        total_tp = total_fp = total_fn = 0.0

        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            x = normalize_batch(x, MEAN_13, STD_13)

            logits = model(x)
            probs = torch.sigmoid(logits)
            preds = (probs >= thr).float()

            total_tp += (preds * y).sum().item()
            total_fp += (preds * (1 - y)).sum().item()
            total_fn += ((1 - preds) * y).sum().item()

        precision = (total_tp + eps) / (total_tp + total_fp + eps)
        recall    = (total_tp + eps) / (total_tp + total_fn + eps)
        f1        = (2 * precision * recall) / (precision + recall + eps)

        log.append((float(thr), f1))

        if f1 > best_f1:
            best_f1 = f1
            best_thr = float(thr)

    return best_thr, best_f1, log

t_best, best_f1, thr_log = find_best_thr_global_f1(model, val_loader)

print("Chosen threshold:", t_best)
print("Best global F1:", best_f1)

print("Top-5 thresholds:")
for thr, f1 in sorted(thr_log, key=lambda x: x[1], reverse=True)[:5]:
    print(thr, f1)

Chosen threshold: 0.05
Best global F1: 0.7470956839432774
Top-5 thresholds:
0.05 0.7470956839432774
0.1 0.7373985763685816
0.15 0.7241672544718237
0.2 0.7107105423100464
0.25 0.6981239255855264


In [23]:
va_loss, va_prec, va_rec, va_f1, va_acc, va_auc = validate_epoch(
    model, val_loader, thr=t_best
)
print(f"val_loss   : {va_loss:.4f}")
print(f"val_prec   : {va_prec:.4f}")
print(f"val_recall : {va_rec:.4f}")
print(f"val_f1     : {va_f1:.4f}")
print(f"val_acc    : {va_acc:.4f}")
print(f"val_PRauc    : {va_auc:.4f}")
print(f"threshold  : {t_best:.2f}")

val_loss   : 0.1087
val_prec   : 0.7820
val_recall : 0.7152
val_f1     : 0.7471
val_acc    : 0.9768
val_auc    : 0.7659
threshold  : 0.05


In [24]:
test_loss, test_prec, test_rec, test_f1, test_acc, test_auc = validate_epoch(
    model, test_loader, thr=t_best
)

print(f"Test Loss   : {test_loss:.4f}")
print(f"Test Prec   : {test_prec:.4f}")
print(f"Test Recall : {test_rec:.4f}")
print(f"Test F1     : {test_f1:.4f}")
print(f"Test Acc    : {test_acc:.4f}")
print(f"Test PRauc    : {test_auc:.4f}")

Test Loss   : 0.0443
Test Prec   : 0.6715
Test Recall : 0.8064
Test F1     : 0.7328
Test Acc    : 0.9857
Test PRauc    : 0.7914


In [25]:
# sweet spot refine
sweet_grid = np.linspace(0.0, 0.10, 11)  # step 0.01
t_best, best_f1, thr_log = find_best_thr_global_f1(model, val_loader,  grid=sweet_grid)
print("Chosen threshold:", t_best)
print("Best global F1:", best_f1)

Chosen threshold: 0.05
Best global F1: 0.7470956839432774


# PRETRAINED

In [14]:
cfg.EPOCHS = 40
print(cfg.EPOCHS)

40


In [17]:
ckpt = torch.load(
    "/kaggle/input/efficient-b4/pytorch/default/1/efficientb4.pth",
    map_location=DEVICE,
    weights_only=False  # biar load semua, bukan cuma weights
)
model.load_state_dict(ckpt["model_state"])

<All keys matched successfully>

In [18]:
from torch.amp import autocast, GradScaler
from sklearn.metrics import average_precision_score
from sklearn.metrics import roc_auc_score
import time

optimizer = torch.optim.Adam(model.parameters(), lr=cfg.LR)
scaler = GradScaler(enabled=(DEVICE.type == "cuda"))

def train_one_epoch(model, loader, thr=0.05, eps=1e-6):
    model.train()
    total_loss_sum = 0.0
    total_samples = 0

    total_tp = total_fp = total_fn = total_tn = 0.0
    all_probs, all_targets = [], []

    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        x = normalize_batch(x, MEAN_13, STD_13)

        optimizer.zero_grad(set_to_none=True)

        with autocast("cuda", enabled=(DEVICE.type == "cuda")):
            logits = model(x)
            loss = criterion(logits, y)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        bs = x.size(0)
        total_loss_sum += loss.item() * bs
        total_samples += bs

        probs = torch.sigmoid(logits)
        preds = (probs >= thr).float()

        total_tp += (preds * y).sum().item()
        total_fp += (preds * (1 - y)).sum().item()
        total_fn += ((1 - preds) * y).sum().item()
        total_tn += ((1 - preds) * (1 - y)).sum().item()

        all_probs.append(probs.flatten().detach().cpu())
        all_targets.append(y.flatten().detach().cpu())

    precision = (total_tp + eps) / (total_tp + total_fp + eps)
    recall    = (total_tp + eps) / (total_tp + total_fn + eps)
    f1        = (2 * precision * recall) / (precision + recall + eps)
    acc       = (total_tp + total_tn + eps) / (
        total_tp + total_tn + total_fp + total_fn + eps
    )

    all_probs = torch.cat(all_probs).numpy()
    all_targets = torch.cat(all_targets).numpy()
    auc = average_precision_score(all_targets, all_probs)

    loss_global = total_loss_sum / total_samples

    return loss_global, precision, recall, f1, acc, auc


@torch.no_grad()
def validate_epoch(model, loader, thr=0.05, eps=1e-6):
    model.eval()
    total_loss_sum = 0.0
    total_samples = 0

    total_tp = total_fp = total_fn = total_tn = 0.0
    all_probs, all_targets = [], []

    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        x = normalize_batch(x, MEAN_13, STD_13)

        logits = model(x)
        loss = criterion(logits, y)

        bs = x.size(0)
        total_loss_sum += loss.item() * bs
        total_samples += bs

        probs = torch.sigmoid(logits)
        preds = (probs >= thr).float()

        total_tp += (preds * y).sum().item()
        total_fp += (preds * (1 - y)).sum().item()
        total_fn += ((1 - preds) * y).sum().item()
        total_tn += ((1 - preds) * (1 - y)).sum().item()

        all_probs.append(probs.flatten().cpu())
        all_targets.append(y.flatten().cpu())

    precision = (total_tp + eps) / (total_tp + total_fp + eps)
    recall    = (total_tp + eps) / (total_tp + total_fn + eps)
    f1        = (2 * precision * recall) / (precision + recall + eps)
    acc       = (total_tp + total_tn + eps) / (
        total_tp + total_tn + total_fp + total_fn + eps
    )

    all_probs = torch.cat(all_probs).numpy()
    all_targets = torch.cat(all_targets).numpy()
    auc = average_precision_score(all_targets, all_probs)

    loss_global = total_loss_sum / total_samples

    return loss_global, precision, recall, f1, acc, auc

In [19]:
BEST_PATH = "/kaggle/working/efficientb4_pretrained.pth"
best_val_f1 = -1

import pandas as pd
history = []

for epoch in range(1, cfg.EPOCHS + 1):
    t0 = time.time()

    tr_loss, tr_prec, tr_rec, tr_f1, tr_acc, tr_auc = train_one_epoch(model, train_loader)
    va_loss, va_prec, va_rec, va_f1, va_acc, va_auc = validate_epoch(model, val_loader)
    elapsed = time.time() - t0

    print(
        f"[unet_basic] Epoch {epoch:03d} | "
        f"tr_loss={tr_loss:.4f} tr_f1={tr_f1:.4f} tr_acc={tr_acc:.4f} tr_PRauc={tr_auc:.4f} || "
        f"va_loss={va_loss:.4f} va_f1={va_f1:.4f} va_acc={va_acc:.4f} va_PRauc={va_auc:.4f} | "
        f"{elapsed:.1f}s"
    )

    history.append({
        "epoch": epoch,
    
        "tr_loss": tr_loss,
        "tr_precision": tr_prec,
        "tr_recall": tr_rec,
        "tr_f1": tr_f1,
        "tr_acc": tr_acc,
        "tr_PRauc": tr_auc,
    
        "va_loss": va_loss,
        "va_precision": va_prec,
        "va_recall": va_rec,
        "va_f1": va_f1,
        "va_acc": va_acc,
        "va_PRauc": va_auc,
    
        "seconds": elapsed
    })

    if va_f1 > best_val_f1:
        torch.save({
            "model_state": model.state_dict(),
            "mean_13": MEAN_13,
            "std_13": STD_13,
            "epoch": epoch,
            "best_val_f1": best_val_f1,
            "va_auc": va_auc,
        }, BEST_PATH)


[unet_basic] Epoch 001 | tr_loss=0.0189 tr_f1=0.8961 tr_acc=0.9865 tr_PRauc=0.9849 || va_loss=0.1292 va_f1=0.6790 va_acc=0.9723 va_PRauc=0.6873 | 232.8s
[unet_basic] Epoch 002 | tr_loss=0.0191 tr_f1=0.8776 tr_acc=0.9837 tr_PRauc=0.9835 || va_loss=0.0956 va_f1=0.7709 va_acc=0.9782 va_PRauc=0.8138 | 231.2s
[unet_basic] Epoch 003 | tr_loss=0.0160 tr_f1=0.9038 tr_acc=0.9876 tr_PRauc=0.9886 || va_loss=0.1006 va_f1=0.7683 va_acc=0.9786 va_PRauc=0.7941 | 232.3s
[unet_basic] Epoch 004 | tr_loss=0.0134 tr_f1=0.9172 tr_acc=0.9894 tr_PRauc=0.9919 || va_loss=0.0962 va_f1=0.7695 va_acc=0.9779 va_PRauc=0.7977 | 230.0s
[unet_basic] Epoch 005 | tr_loss=0.0146 tr_f1=0.9123 tr_acc=0.9888 tr_PRauc=0.9906 || va_loss=0.1477 va_f1=0.7005 va_acc=0.9754 va_PRauc=0.7236 | 229.5s
[unet_basic] Epoch 006 | tr_loss=0.0178 tr_f1=0.9054 tr_acc=0.9878 tr_PRauc=0.9865 || va_loss=0.1568 va_f1=0.6519 va_acc=0.9720 va_PRauc=0.6843 | 229.9s
[unet_basic] Epoch 007 | tr_loss=0.0134 tr_f1=0.9135 tr_acc=0.9889 tr_PRauc=0.9923

KeyboardInterrupt: 

In [20]:
df_history = pd.DataFrame(history)

df_history = df_history.round(4)
csv_path = "/kaggle/working/training_metrics_eff_pretrained.csv"
df_history.to_csv(csv_path, index=False)
print("Saved to:", csv_path)
df_history

Saved to: /kaggle/working/training_metrics_eff_pretrained.csv


,epoch,tr_loss,tr_precision,tr_recall,tr_f1,tr_acc,tr_PRauc,va_loss,va_precision,va_recall,va_f1,va_acc,va_PRauc,seconds
0,1,0.0189,0.8204,0.9873,0.8961,0.9865,0.9849,0.1292,0.7623,0.6120,0.6790,0.9723,0.6873,232.7803
1,2,0.0191,0.7871,0.9916,0.8776,0.9837,0.9835,0.0956,0.7747,0.7671,0.7709,0.9782,0.8138,231.2160
2,3,0.0160,0.8296,0.9927,0.9038,0.9876,0.9886,0.1006,0.7968,0.7418,0.7683,0.9786,0.7941,232.2950
3,4,0.0134,0.8509,0.9947,0.9172,0.9894,0.9919,0.0962,0.7683,0.7706,0.7695,0.9779,0.7977,229.9897
4,5,0.0146,0.8434,0.9935,0.9123,0.9888,0.9906,0.1477,0.8383,0.6017,0.7005,0.9754,0.7236,229.5133
5,6,0.0178,0.8352,0.9884,0.9054,0.9878,0.9865,0.1568,0.8030,0.5486,0.6519,0.9720,0.6843,229.9425
6,7,0.0134,0.8456,0.9932,0.9135,0.9889,0.9923,0.1202,0.8489,0.6878,0.7599,0.9792,0.7868,232.0495
7,8,0.0107,0.8786,0.9960,0.9336,0.9917,0.9949,0.0818,0.7297,0.8159,0.7704,0.9767,0.8196,230.9640
8,9,0.0222,0.8841,0.9791,0.9292,0.9912,0.9843,0.1328,0.8284,0.6191,0.7086,0.9756,0.7302,231.2039
9,10,0.0148,0.8385,0.9896,0.9078,0.9882,0.9906,0.1122,0.7897,0.7459,0.7672,0.9783,0.7827,229.6631


### TES 2 (MEAN PER PATCH)

In [17]:
from torch.amp import autocast, GradScaler
from sklearn.metrics import average_precision_score
import torch
import time

optimizer = torch.optim.Adam(model.parameters(), lr=cfg.LR)
scaler = GradScaler(enabled=(DEVICE.type == "cuda"))

def train_one_epoch(model, loader, thr=0.5, eps=1e-6):
    model.train()
    patch_losses, patch_f1s, patch_accs, patch_aps = [], [], [], []

    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        x = normalize_batch(x, MEAN_13, STD_13)

        optimizer.zero_grad(set_to_none=True)

        with autocast("cuda", enabled=(DEVICE.type == "cuda")):
            logits = model(x)
            loss = criterion(logits, y)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        probs = torch.sigmoid(logits)
        preds = (probs >= thr).float()

        # per patch metrics
        for p, t, prob in zip(preds, y, probs):
            tp = (p * t).sum().item()
            fp = (p * (1 - t)).sum().item()
            fn = ((1 - p) * t).sum().item()
            tn = ((1 - p) * (1 - t)).sum().item()

            precision = (tp + eps) / (tp + fp + eps)
            recall    = (tp + eps) / (tp + fn + eps)
            f1        = (2 * precision * recall) / (precision + recall + eps)
            acc       = (tp + tn + eps) / (tp + tn + fp + fn + eps)

            patch_f1s.append(f1)
            patch_accs.append(acc)
            patch_losses.append(criterion(p.unsqueeze(0), t.unsqueeze(0)).item())
            
            # handle patch tanpa positif supaya warning hilang (Kalau ada, baru hitung, kalau nggak, langsung append 0)
            y_np = t.flatten().cpu().numpy()
            prob_np = prob.flatten().detach().cpu().numpy()
            if y_np.sum() > 0:
                patch_aps.append(average_precision_score(y_np, prob_np))
            else:
                patch_aps.append(0.0)

    # mean per patch
    loss_mean_patch = sum(patch_losses) / len(patch_losses)
    f1_mean_patch = sum(patch_f1s) / len(patch_f1s)
    acc_mean_patch = sum(patch_accs) / len(patch_accs)
    ap_mean_patch = sum(patch_aps) / len(patch_aps)

    return loss_mean_patch, f1_mean_patch, acc_mean_patch, ap_mean_patch


@torch.no_grad()
def validate_epoch(model, loader, thr=0.5, eps=1e-6):
    model.eval()
    patch_losses, patch_f1s, patch_accs, patch_aps = [], [], [], []

    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        x = normalize_batch(x, MEAN_13, STD_13)

        logits = model(x)
        loss = criterion(logits, y)

        probs = torch.sigmoid(logits)
        preds = (probs >= thr).float()

        # per patch metrics
        for p, t, prob in zip(preds, y, probs):
            tp = (p * t).sum().item()
            fp = (p * (1 - t)).sum().item()
            fn = ((1 - p) * t).sum().item()
            tn = ((1 - p) * (1 - t)).sum().item()

            precision = (tp + eps) / (tp + fp + eps)
            recall    = (tp + eps) / (tp + fn + eps)
            f1        = (2 * precision * recall) / (precision + recall + eps)
            acc       = (tp + tn + eps) / (tp + tn + fp + fn + eps)

            patch_f1s.append(f1)
            patch_accs.append(acc)
            patch_losses.append(criterion(p.unsqueeze(0), t.unsqueeze(0)).item())
            
            y_np = t.flatten().cpu().numpy()
            prob_np = prob.flatten().detach().cpu().numpy()
            if y_np.sum() > 0:
                patch_aps.append(average_precision_score(y_np, prob_np))
            else:
                patch_aps.append(0.0)

    loss_mean_patch = sum(patch_losses) / len(patch_losses)
    f1_mean_patch = sum(patch_f1s) / len(patch_f1s)
    acc_mean_patch = sum(patch_accs) / len(patch_accs)
    ap_mean_patch = sum(patch_aps) / len(patch_aps)

    return loss_mean_patch, f1_mean_patch, acc_mean_patch, ap_mean_patch

In [18]:
BEST_PATH = "/kaggle/working/efficientb4_mean.pth"
best_val_f1 = -1

import pandas as pd
history = []

for epoch in range(1, cfg.EPOCHS + 1):
    t0 = time.time()

    tr_loss, tr_f1, tr_acc, tr_ap = train_one_epoch(model, train_loader)
    va_loss, va_f1, va_acc, va_ap = validate_epoch(model, val_loader, thr=0.5)
    elapsed = time.time() - t0

    print(
        f"[unet_basic] Epoch {epoch:03d} | "
        f"tr_loss={tr_loss:.4f} tr_f1={tr_f1:.4f} tr_acc={tr_acc:.4f} tr_PRauc={tr_ap:.4f} || "
        f"va_loss={va_loss:.4f} va_f1={va_f1:.4f} va_acc={va_acc:.4f} va_PRauc={va_ap:.4f} | "
        f"{elapsed:.1f}s"
    )

    history.append({
        "epoch": epoch,
        "tr_loss": tr_loss,
        "tr_f1": tr_f1,
        "tr_acc": tr_acc,
        "tr_PRauc": tr_ap,
        "va_loss": va_loss,
        "va_f1": va_f1,
        "va_acc": va_acc,
        "va_PRauc": va_ap,
        "seconds": elapsed
    })

    if va_f1 > best_val_f1:
        torch.save({
            "model_state": model.state_dict(),
            "mean_13": MEAN_13,
            "std_13": STD_13,
            "epoch": epoch,
            "best_val_f1": va_f1,
            "va_ap": va_ap,
        }, BEST_PATH)
        best_val_f1 = va_f1

[unet_basic] Epoch 001 | tr_loss=0.6983 tr_f1=0.3795 tr_acc=0.9330 tr_PRauc=0.0681 || va_loss=0.6932 va_f1=0.8381 va_acc=0.9522 va_PRauc=0.0607 | 126.7s
[unet_basic] Epoch 002 | tr_loss=0.6932 tr_f1=0.6936 tr_acc=0.9411 tr_PRauc=0.0920 || va_loss=0.6931 va_f1=0.8462 va_acc=0.9522 va_PRauc=0.0716 | 124.7s
[unet_basic] Epoch 003 | tr_loss=0.6931 tr_f1=0.7295 tr_acc=0.9411 tr_PRauc=0.1172 || va_loss=0.6931 va_f1=0.8543 va_acc=0.9522 va_PRauc=0.0695 | 124.7s
[unet_basic] Epoch 004 | tr_loss=0.6931 tr_f1=0.6901 tr_acc=0.9414 tr_PRauc=0.1413 || va_loss=0.6931 va_f1=0.8465 va_acc=0.9522 va_PRauc=0.0770 | 124.1s
[unet_basic] Epoch 005 | tr_loss=0.6929 tr_f1=0.7101 tr_acc=0.9473 tr_PRauc=0.1567 || va_loss=0.6931 va_f1=0.8381 va_acc=0.9522 va_PRauc=0.0751 | 124.7s
[unet_basic] Epoch 006 | tr_loss=0.6900 tr_f1=0.7297 tr_acc=0.9554 tr_PRauc=0.1601 || va_loss=0.6917 va_f1=0.8178 va_acc=0.9630 va_PRauc=0.0889 | 124.8s
[unet_basic] Epoch 007 | tr_loss=0.6875 tr_f1=0.7710 tr_acc=0.9631 tr_PRauc=0.1746

KeyboardInterrupt: 

In [22]:
t_best = 0.05

In [23]:
va_loss, va_prec, va_rec, va_f1, va_acc, va_auc = validate_epoch(
    model, val_loader, thr=t_best
)
print(f"val_loss   : {va_loss:.4f}")
print(f"val_prec   : {va_prec:.4f}")
print(f"val_recall : {va_rec:.4f}")
print(f"val_f1     : {va_f1:.4f}")
print(f"val_acc    : {va_acc:.4f}")
print(f"val_PRauc    : {va_auc:.4f}")
print(f"threshold  : {t_best:.2f}")

val_loss   : 0.1387
val_prec   : 0.8117
val_recall : 0.6405
val_f1     : 0.7160
val_acc    : 0.9757
val_PRauc    : 0.7630
threshold  : 0.05


In [26]:
test_loss, test_prec, test_rec, test_f1, test_acc, test_auc = validate_epoch(
    model, test_loader, thr=t_best
)

print(f"Test Loss   : {test_loss:.4f}")
print(f"Test Prec   : {test_prec:.4f}")
print(f"Test Recall : {test_rec:.4f}")
print(f"Test F1     : {test_f1:.4f}")
print(f"Test Acc    : {test_acc:.4f}")
print(f"Test PRauc  : {test_auc:.4f}")
print(f"threshold   : {t_best:.2f}")

Test Loss   : 0.0501
Test Prec   : 0.7206
Test Recall : 0.7792
Test F1     : 0.7488
Test Acc    : 0.9873
Test PRauc  : 0.7971
threshold   : 0.05
